In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import mlflow

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (root_mean_squared_error, mean_absolute_error, r2_score)


df = pd.read_csv("D:\\day1_mlops\\data\\data.csv")
df.head()


,Unnamed: 0,TV,radio,newspaper,sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


In [2]:
x = df[["TV", "radio","newspaper"]]
y = df["sales"]

xtrain, xtest, ytrain, ytest = train_test_split(x,y, test_size=0.2, random_state=56)


In [3]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

In [4]:
mlflow.set_experiment("Advertising Sale Prediction")

2026/09/11 01:00:53 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/11 01:00:55 INFO mlflow.store.db.utils: Updating database tables
2026/09/11 01:02:22 INFO mlflow.tracking.fluent: Experiment with name 'Advertising Sale Prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:d:/day1_mlops/notebooks/mlruns/1', creation_time=1789068742895, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789068742895, lifecycle_stage='active', name='Advertising Sale Prediction', tags={}, trace_location=None, workspace='default'>

In [5]:
# Now, we create a run inside the experiment Adverstising Sales Prediction


with mlflow.start_run(run_name="Linear Regression"):

  model = LinearRegression()

  model.fit(xtrain, ytrain)

  y_pred = model.predict(xtest)

  rmse = root_mean_squared_error(ytest, y_pred)
  r2 = r2_score(ytest, y_pred)


  # Parameters

  mlflow.log_param("model_type", "LinearRegression")
  mlflow.log_param("test_size", 0.2)
  mlflow.log_param("random_state", 56)

  # Metrics

  mlflow.log_metric("test_rmse", rmse)
  mlflow.log_metric("test_r2", r2)

In [6]:
# Now , we create another run using a Ridge Regression model

with mlflow.start_run(run_name="Ridge Regression"):

  model = Ridge(alpha=1.0) 
  model.fit(xtrain, ytrain)

  ypred = model.predict(xtest)

  rmse = root_mean_squared_error(ytest, y_pred)
  r2 = r2_score(ytest, y_pred) 

  # Log the parameters

  mlflow.log_param("model_type", "Ridge Regression")
  mlflow.log_param("alpha", 1.0)
  mlflow.log_param("test_siz", 0.2)
  mlflow.log_param("random_state", 56)

  # Log the metrics

  mlflow.log_metric("test_rmse", rmse)
  mlflow.log_metric("test_r2", r2)

  # Log Artifacts

  mlflow.sklearn.log_model(sk_model = model, name="Ridge_Reg_Model")


2026/09/11 01:11:10 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\SUREND~1\AppData\Local\Temp\tmp4ymcrr9n\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.9.0', 'skops==0.14.0']. Set logging level to DEBUG to see the full traceback. 


In [7]:
# Turn on Scikit-learn autologging

mlflow.sklearn.autolog()


with mlflow.start_run(run_name="Random Forest Autolog") as run:


    model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

    model.fit(xtrain, ytrain)

    test_pred = model.predict(xtest)

    test_rmse = root_mean_squared_error(ytest, test_pred)
    test_mae = mean_absolute_error(ytest, test_pred)
    test_r2 = r2_score(ytest, test_pred)

    # Custom project metrics

    mlflow.log_metrics({
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2
    })

2026/09/11 01:12:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/09/11 01:14:33 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\SUREND~1\AppData\Local\Temp\tmpuyst2oed\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.9.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


Registering the Model

The MLflow Model Registry is a central hub (like an app store or catlog) to keep all production-ready models in one shared, searchable place instead of scattered across folders or runs.

Among all the experiments you perform, register the final selected model.

In a production environment , the models are continously trained. This means the registered models would have many vers
Advertising_Sales_Model 
| 
|__ version 1 
|__ version 2 
|__ version 3 
|__ version 4


In [8]:
# For registering the model, we require the model URI.
# URI is a unique identifier for a model.

run_id = run.info.run_id
model_uri = f"runs:/{run_id}/model"
print(model_uri)

runs:/44267aee2b064040b728171764aabc0d/model


In [9]:
# Now let's register the model
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="Advertising_Sale_Model"
)

Successfully registered model 'Advertising_Sale_Model'.
2026/09/11 01:15:30 WARNING mlflow.tracking._model_registry.fluent: Run with id 44267aee2b064040b728171764aabc0d has no artifacts at artifact path 'model', registering model based on models:/m-6492117d7c5343ef9e562dbd5014980f instead
Created version '1' of model 'Advertising_Sale_Model'.


In [10]:
from mlflow import MlflowClient

Client = MlflowClient()

# 1. Assign an alias to a specific version.
# (Sets the slias 'champion' to Version 2 of 'fraud_detector')

Client.set_registered_model_alias(
    name="Advertising_Sale_Model",
    alias="champion",
    version="1"
)

In [12]:
model = mlflow.sklearn.load_model(
    "models:/Advertising_Sale_Model@champion"
)

# Generate new predicions

new_data = pd.DataFrame({
    "TV": [150.0],
    "radio": [25.0],
    "newspaper": [30.0]
}) 
print(model.predict(new_data))

[15.13020624]
